In [1]:
import trafilatura as trf
from collections.abc import Sequence, Mapping
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

In [2]:
# Install ollama.
# Download gemma4:e4b using "ollama pull gemma4:e4b"
class LocalLLM:
    def __init__(
        self,
        model: str='gemma4:e4b',
        temperature: float=0.7,
    ):
        self._model = model
        self._temperature = temperature
        
    def __call__(self):
        return ChatOllama(
            model=self._model,
            temperature=self._temperature,
            validate_model_on_init=True,
        )

llm = LocalLLM()        

In [3]:
def extract(url: str) -> str:
    return trf.extract(trf.fetch_url(url))    

In [4]:
list_of_urls = [
    'https://www.nationalgallery.org.uk/paintings/learn-about-art/guide-to-impressionism',
    'https://www.tate.org.uk/art/art-terms/i/impressionism',
    'https://www.artic.edu/highlights/5/impressionism',
    'https://en.wikipedia.org/wiki/Impressionism',
]

texts = [extract(url) for url in list_of_urls]

In [5]:
doc_summary_template = '''
Write a concise summary of the following text:
{text}
DOC SUMMARY:
'''
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)
doc_summary_chain = doc_summary_prompt | llm() | StrOutputParser()


refine_summary_template = '''
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far:
{current_refined_summary}

This is the content of the additional document:
{text}

Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is.
'''

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)
refine_chain = refine_summary_prompt | llm() | StrOutputParser()

In [6]:
def refine_summary(docs):
    intermediate_steps = []
    current_refined_summary = doc_summary_chain.invoke({
        'text': docs[0],
    })
        
    for doc in docs[1:]:
        intermediate_step = {
            'current_refined_summary': current_refined_summary, 
            'text': doc
        }
        intermediate_steps.append(intermediate_step)        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {
        'final_summary': current_refined_summary,
        'intermediate_steps': intermediate_steps
    }

In [7]:
full_summary = refine_summary(texts)

In [8]:
print(full_summary['intermediate_steps'][0]['current_refined_summary'])

Impressionism originated in 1874 when a group of rejected artists (including Monet, Renoir, and Degas) defiantly held their own exhibition. The movement is characterized by painting modern life—such as landscapes and urban scenes—using bright, pure colors. Technically, Impressionists are known for painting outdoors (*en plein air*) and employing visible, rapidly applied brushstrokes. Though initially considered radical, Impressionist paintings are now among the most popular and celebrated art forms.


In [9]:
print(full_summary['intermediate_steps'][1]['current_refined_summary'])

Impressionism was developed by Parisian artists, including Claude Monet, beginning in the early 1860s. The movement formally began with the first group exhibition in Paris in 1874, featuring works by artists such as Monet, Auguste Renoir, Edgar Degas, Paul Cézanne, Camille Pissarro, Berthe Morisot, and Édouard Manet.

The movement is characterized by painting modern life—including landscapes and urban scenes—using bright, pure colors. Technically, Impressionists abandoned the studio to work outdoors (*en plein air*). This method allowed them to capture the momentary and transient effects of sunlight, leading to visible, rapidly applied brushstrokes that appear broken into separate dabs.

The movement’s name derived from Monet’s painting, *Impression, Sunrise*, which was initially used by critics as an insult. Though initially viewed with derision, Impressionist paintings are now celebrated globally. While originating in France, the movement had a wide influence, with core British Impre

In [10]:
print(full_summary['intermediate_steps'][2]['current_refined_summary'])

Impressionism was a revolutionary art movement developed by Parisian artists, notably Claude Monet, beginning in the early 1860s. It formally debuted with the first group exhibition in Paris in 1874, featuring key figures such as Monet, Pierre-August Renoir, Edgar Degas, Camille Pissarro, Berthe Morisot, and Édouard Manet.

The movement is fundamentally characterized by its subject matter—the depiction of modern, everyday Parisian life, including bustling urban scenes, leisurely picnics, and landscapes. Technically, Impressionists abandoned the confines of the studio to work outdoors (*en plein air*). This method allowed them to capture the fleeting, momentary effects of sunlight, resulting in visible, rapidly applied brushstrokes that appear broken into separate dabs of color. The movement's name originated from Monet’s painting, *Impression, Sunrise*, which, despite initial derision, cemented the style's identity.

The artists explored modern life through distinct lenses:

*   **Leis

In [11]:
print(full_summary['final_summary'])

**Impressionism: A Synthesis Summary**

Impressionism was a revolutionary art movement developed by Parisian artists, notably Claude Monet, beginning in the early 1860s. It formally debuted with the first group exhibition in Paris in 1874, featuring key figures such as Monet, Pierre-Auguste Renoir, Edgar Degas, Camille Pissarro, Berthe Morisot, and Édouard Manet. The movement was fundamentally a rebellion against the conservative standards of the academic art establishment (the *Académie des Beaux-Arts*), which favored historical, mythological, and highly finished, detailed works.

**Core Characteristics and Technique**

Technically, Impressionists abandoned the confines of the studio to work outdoors (*en plein air*). This radical shift allowed them to capture the fleeting, momentary effects of light and color. They broke with tradition by:

*   **Brushwork:** Using visible, rapidly applied, and short "broken" brushstrokes, which were applied side by side with minimal blending. This t